# 05b - Mental Model Comparison (Struggling Student 10155)

This notebook runs a controlled single-student comparison for the **struggling cluster medoid (10155)** using the same experiment design as the average-student run:
- Condition A: Curriculum-Aware prompt (no mental model)
- Condition B: Curriculum-Aware prompt + student mental model payload

Target student: 10155.

In [ ]:
import json
import time
from pathlib import Path

import pandas as pd
from google.genai import types

ROOT = Path.cwd()
if not (ROOT / 'lib').exists() and (ROOT.parent / 'lib').exists():
    ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.experiment_utils import create_client, load_best_attempts_df
from lib.llm_batch_analyzer import format_submissions, clean_json_response
from lib.mental_model import load_skill_map, calculate_student_profile, build_prerequisite_graph, get_weak_skills, get_prereq_risk_chains, build_mental_model_payload

from lib.prompts import build_curriculum_aware_prompt, KC_TAGS
from utils.dataset import load_topics_json, load_problem_descriptions

MODEL_ID = 'gemini-2.5-flash'
TARGET_STUDENT_ID = 10155  # Struggling cluster medoid (was 14359)
NUM_PROBLEMS = 12
RANDOM_SEED = 42
SLEEP_SECONDS = 1.0

client = create_client()
print(f'Ready. Model={MODEL_ID}, Student={TARGET_STUDENT_ID}, Requested problems={NUM_PROBLEMS}')

Ready. Model=gemini-2.5-flash, Student=10155, Requested problems=12


In [2]:
best_attempts_df = load_best_attempts_df()
student_df = best_attempts_df[best_attempts_df['SubjectID'] == TARGET_STUDENT_ID].copy()
student_df = student_df.drop_duplicates(subset=['ProblemID']).copy()

if student_df.empty:
    raise ValueError(f'No submissions found for student {TARGET_STUDENT_ID}.')

score_max = float(student_df['Score'].max())
if score_max <= 1.0:
    student_df['ScorePct'] = student_df['Score'] * 100.0
else:
    student_df['ScorePct'] = student_df['Score']

available_n = len(student_df)
target_n = min(NUM_PROBLEMS, available_n)

if available_n < 10:
    print(f'Fallback: student has only {available_n} problems (<10). Using all available problems.')
    selected_df = student_df.sample(n=target_n, random_state=RANDOM_SEED).copy()
else:
    full_df = student_df[student_df['ScorePct'] >= 99.9].copy()
    partial_df = student_df[(student_df['ScorePct'] > 0.0) & (student_df['ScorePct'] < 99.9)].copy()
    zero_df = student_df[student_df['ScorePct'] <= 0.0].copy()

    base_quota = target_n // 3
    remainder = target_n - (base_quota * 3)
    quotas = {'full': base_quota, 'partial': base_quota, 'zero': base_quota}
    for key in ['partial', 'full', 'zero'][:remainder]:
        quotas[key] += 1

    picked = []
    bucket_map = {'full': full_df, 'partial': partial_df, 'zero': zero_df}
    for name, bucket in bucket_map.items():
        take_n = min(quotas[name], len(bucket))
        if take_n > 0:
            picked.append(bucket.sample(n=take_n, random_state=RANDOM_SEED))

    selected_df = pd.concat(picked, ignore_index=False).drop_duplicates(subset=['ProblemID']) if picked else pd.DataFrame(columns=student_df.columns)

    remaining_needed = target_n - len(selected_df)
    if remaining_needed > 0:
        remaining_pool = student_df[~student_df['ProblemID'].isin(selected_df['ProblemID'])].copy()
        if len(remaining_pool) > 0:
            topup = remaining_pool.sample(n=min(remaining_needed, len(remaining_pool)), random_state=RANDOM_SEED)
            selected_df = pd.concat([selected_df, topup], ignore_index=False)

selected_df = selected_df.sort_values(by=['ScorePct', 'ProblemID'], ascending=[False, True]).head(target_n).copy()
selected_df = selected_df.reset_index(drop=True)

print(f'Available problems for student {TARGET_STUDENT_ID}: {available_n}')
print(f'Selected problems: {len(selected_df)}')
print('Score bucket counts in selected set:')
print(selected_df.assign(bucket=selected_df['ScorePct'].apply(lambda x: 'full' if x >= 99.9 else ('zero' if x <= 0.0 else 'partial')))['bucket'].value_counts())
display(selected_df[['ProblemID', 'Score', 'ScorePct']])

Main table: 201,570 rows
CodeState table: 69,627 rows
Subject table: 381 rows
Joined dataset: 191,584 rows
Best attempts: 15,375 rows (372 students, 50 problems)
Available problems for student 10155: 46
Selected problems: 12
Score bucket counts in selected set:
bucket
partial    6
full       4
zero       2
Name: count, dtype: int64


,ProblemID,Score,ScorePct
0,1,1.000000,100.0000
1,20,1.000000,100.0000
2,232,1.000000,100.0000
3,233,1.000000,100.0000
4,235,0.857143,85.7143
5,236,0.750000,75.0000
6,106,0.625000,62.5000
7,46,0.555556,55.5556
8,36,0.411765,41.1765
9,49,0.411765,41.1765


In [3]:
skill_map, all_skills = load_skill_map()
student_profile = calculate_student_profile(TARGET_STUDENT_ID, best_attempts_df, skill_map, all_skills)
weak_skills = get_weak_skills(student_profile, threshold=0.6)
G = build_prerequisite_graph()
risk_chains = get_prereq_risk_chains(G, weak_skills, max_items=8)

mental_model = build_mental_model_payload(
    student_id=TARGET_STUDENT_ID,
    profile=student_profile,
    weak_skill_pairs=weak_skills,
    graph=G,
)

print(f'Profile skills tracked: {len(student_profile)}')
print(f'Weak skills identified: {len(weak_skills)}')
print('Top weak skills:')
for skill, score in weak_skills[:8]:
    print(f'  - {skill}: {score:.3f}')
print('')
print(f'Risk chains found: {len(risk_chains)}')

Profile skills tracked: 18
Weak skills identified: 14
Top weak skills:
  - While: 0.044
  - DefFunction: 0.216
  - StringConcat: 0.221
  - Math%: 0.297
  - StringLen: 0.364
  - StringIndex: 0.400
  - StringEqual: 0.417
  - ArrayIndex: 0.438

Risk chains found: 8


In [4]:
topics = load_topics_json() or {}
problem_descriptions = load_problem_descriptions() or {}
selected_problem_ids = [int(pid) for pid in selected_df['ProblemID'].tolist()]

baseline_prompt = build_curriculum_aware_prompt(
    topics=topics,
    problems=problem_descriptions,
    focus_problem_ids=selected_problem_ids,
)

print(f'Baseline prompt prepared for {len(selected_problem_ids)} problems.')
print('Enriched prompt will be created in the helper cell using the same baseline prompt + mental model JSON payload.')

Baseline prompt prepared for 12 problems.
Enriched prompt will be created in the helper cell using the same baseline prompt + mental model JSON payload.


In [5]:
EXACT_KC_TAGS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]
VALID_KC_SET = set(EXACT_KC_TAGS)

if set(KC_TAGS) != VALID_KC_SET:
    print('Warning: imported KC_TAGS differs from the required exact KC vocabulary. Validation will use EXACT_KC_TAGS.')

prompts_df = pd.read_csv(ROOT / 'dataset/CodeWorkout/Problem_Prompts/problem_prompts.csv')

def add_mental_model_context(base_prompt: str, mental_model_payload: dict) -> str:
    context = json.dumps(mental_model_payload, indent=2)
    return (
        base_prompt
        + "\n\nAdditional Student Mental Model Context:\n"
        + context
        + "\n\nUse this context to better judge likely misconceptions and future risks."
    )

def run_one_call(submission_row, system_instruction: str):
    formatted_input = format_submissions([submission_row.to_dict()])
    t0 = time.time()
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=formatted_input,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
                response_mime_type='application/json',
            ),
        )
        raw_text = response.text if response and response.text else '{}'
        parsed = json.loads(clean_json_response(raw_text))
        return parsed, round(time.time() - t0, 3), None
    except Exception as e:
        return None, round(time.time() - t0, 3), str(e)

def extract_expected_kc_tags(problem_id: int) -> list[str]:
    row = prompts_df[prompts_df['ProblemID'] == int(problem_id)]
    if row.empty:
        return []
    row = row.iloc[0]
    tags = []
    for tag in EXACT_KC_TAGS:
        value = row.get(tag, 0)
        if pd.notna(value) and float(value) == 1.0:
            tags.append(tag)
    return tags

def extract_kc_tags(output_obj) -> tuple[list[str], list[str]]:
    """Extract KC tags from Curriculum-Aware LLM output structure."""
    if not isinstance(output_obj, dict):
        return [], []

    valid_tags = set()
    invalid_tags = set()
    analysis_list = output_obj.get("student_analysis", [])
    if not isinstance(analysis_list, list):
        analysis_list = []

    for analysis in analysis_list:
        if not isinstance(analysis, dict): continue
        for gap in analysis.get("knowledge_gaps", []):
            tag = ""
            if isinstance(gap, dict):
                tag = gap.get("missing_concept", "")
            elif isinstance(gap, str):
                tag = gap
            tag = tag.strip()
            if tag:
                if tag in VALID_KC_SET:
                    valid_tags.add(tag)
                else:
                    invalid_tags.add(tag)

        for pred in analysis.get("future_predictions", []):
            tag = ""
            if isinstance(pred, dict):
                tag = pred.get("at_risk_topic", "")
            elif isinstance(pred, str):
                tag = pred
            tag = tag.strip()
            if tag:
                if tag in VALID_KC_SET:
                    valid_tags.add(tag)
                else:
                    invalid_tags.add(tag)

    for key in ["knowledge_gaps", "future_predictions"]:
        val = output_obj.get(key, [])
        if isinstance(val, list):
            for item in val:
                if isinstance(item, str):
                    item = item.strip()
                    if item in VALID_KC_SET:
                        valid_tags.add(item)
                    elif item:
                        invalid_tags.add(item)

    return sorted(valid_tags), sorted(invalid_tags)

def calculate_overlap(predicted_tags: list[str], expected_tags: list[str]) -> dict:
    pred_set = set(predicted_tags)
    exp_set = set(expected_tags)
    overlap = pred_set & exp_set
    precision = (len(overlap) / len(pred_set)) if pred_set else 0.0
    recall = (len(overlap) / len(exp_set)) if exp_set else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        'pred_count': len(pred_set),
        'expected_count': len(exp_set),
        'overlap_count': len(overlap),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1': round(f1, 4),
    }

enriched_prompt = add_mental_model_context(baseline_prompt, mental_model)
print('Helpers ready. Enriched prompt prepared.')

Helpers ready. Enriched prompt prepared.


In [6]:
weak_skill_names = [s[0] for s in weak_skills]
rows = []
total = len(selected_df)

for i, (_, sub) in enumerate(selected_df.iterrows(), start=1):
    pid = int(sub['ProblemID'])
    score_pct = float(sub['ScorePct'])
    expected_tags = extract_expected_kc_tags(pid)

    baseline_out, baseline_time, baseline_err = run_one_call(sub, baseline_prompt)
    enriched_out, enriched_time, enriched_err = run_one_call(sub, enriched_prompt)

    baseline_tags, baseline_invalid = extract_kc_tags(baseline_out if baseline_err is None else {})
    enriched_tags, enriched_invalid = extract_kc_tags(enriched_out if enriched_err is None else {})

    baseline_gap_count = 0
    enriched_gap_count = 0
    try:
        analysis = baseline_out.get("student_analysis", [{}]) if isinstance(baseline_out, dict) else [{}]
        baseline_gap_count = len(analysis[0].get("knowledge_gaps", [])) if isinstance(analysis, list) and len(analysis) > 0 else 0
    except: pass
    try:
        analysis = enriched_out.get("student_analysis", [{}]) if isinstance(enriched_out, dict) else [{}]
        enriched_gap_count = len(analysis[0].get("knowledge_gaps", [])) if isinstance(analysis, list) and len(analysis) > 0 else 0
    except: pass

    baseline_weak_overlap = calculate_overlap(baseline_tags, weak_skill_names)
    enriched_weak_overlap = calculate_overlap(enriched_tags, weak_skill_names)
    baseline_relevance = calculate_overlap(baseline_tags, expected_tags)
    enriched_relevance = calculate_overlap(enriched_tags, expected_tags)

    is_perfect = score_pct >= 99.9
    baseline_perfect_correct = (baseline_gap_count == 0) if is_perfect else None
    enriched_perfect_correct = (enriched_gap_count == 0) if is_perfect else None

    rows.append({
        'SubjectID': int(sub['SubjectID']),
        'ProblemID': pid,
        'Score': float(sub['Score']),
        'ScorePct': score_pct,
        'IsPerfect': is_perfect,
        'Expected_KCTags': expected_tags,
        'WeakSkills': weak_skill_names,
        'Baseline_GAP_Count': baseline_gap_count,
        'Enriched_GAP_Count': enriched_gap_count,
        'Baseline_WeakOverlap_F1': baseline_weak_overlap['f1'],
        'Enriched_WeakOverlap_F1': enriched_weak_overlap['f1'],
        'Baseline_KCTags': baseline_tags,
        'Enriched_KCTags': enriched_tags,
        'Baseline_TimeSec': baseline_time,
        'Enriched_TimeSec': enriched_time,
        'Baseline_PerfectCorrect': baseline_perfect_correct,
        'Enriched_PerfectCorrect': enriched_perfect_correct,
        'Baseline_Relevance_F1': baseline_relevance['f1'],
        'Enriched_Relevance_F1': enriched_relevance['f1'],
    })

    tag_status = "SAME" if baseline_tags == enriched_tags else "DIFF"
    print(f"[{i}/{total}] Problem {pid} | Score={score_pct:.1f}% | Gaps: B={baseline_gap_count} E={enriched_gap_count} [{tag_status}]")

    if SLEEP_SECONDS > 0: time.sleep(SLEEP_SECONDS)

comparison_df = pd.DataFrame(rows)
print(f"\nCompleted A/B runs: {len(comparison_df)} rows")
display(comparison_df[['ProblemID', 'ScorePct', 'Baseline_GAP_Count', 'Enriched_GAP_Count', 'Baseline_WeakOverlap_F1', 'Enriched_WeakOverlap_F1']])

[1/12] Problem 1 | Score=100.0% | Gaps: B=0 E=0 [SAME]
[2/12] Problem 20 | Score=100.0% | Gaps: B=0 E=0 [SAME]
[3/12] Problem 232 | Score=100.0% | Gaps: B=2 E=2 [DIFF]
[4/12] Problem 233 | Score=100.0% | Gaps: B=0 E=0 [SAME]
[5/12] Problem 235 | Score=85.7% | Gaps: B=3 E=4 [DIFF]
[6/12] Problem 236 | Score=75.0% | Gaps: B=2 E=2 [SAME]
[7/12] Problem 106 | Score=62.5% | Gaps: B=4 E=4 [DIFF]
[8/12] Problem 46 | Score=55.6% | Gaps: B=4 E=3 [DIFF]
[9/12] Problem 36 | Score=41.2% | Gaps: B=4 E=6 [DIFF]
[10/12] Problem 49 | Score=41.2% | Gaps: B=5 E=3 [DIFF]
[11/12] Problem 32 | Score=0.0% | Gaps: B=3 E=3 [DIFF]
[12/12] Problem 101 | Score=0.0% | Gaps: B=3 E=3 [DIFF]

Completed A/B runs: 12 rows


,ProblemID,ScorePct,Baseline_GAP_Count,Enriched_GAP_Count,Baseline_WeakOverlap_F1,Enriched_WeakOverlap_F1
0,1,100.0000,0,0,0.0000,0.0000
1,20,100.0000,0,0,0.0000,0.0000
2,232,100.0000,2,2,0.1176,0.2222
3,233,100.0000,0,0,0.0000,0.0000
4,235,85.7143,3,4,0.1111,0.1176
5,236,75.0000,2,2,0.1176,0.1176
6,106,62.5000,4,4,0.4211,0.6000
7,46,55.5556,4,3,0.5000,0.6000
8,36,41.1765,4,6,0.5714,0.6364
9,49,41.1765,5,3,0.5263,0.6000


In [8]:
print("=" * 70)
print("COMPARISON: Baseline vs Context-Enriched")
print("=" * 70)

# Split into perfect vs imperfect problems
perfect_df = comparison_df[comparison_df['IsPerfect'] == True]
imperfect_df = comparison_df[comparison_df['IsPerfect'] == False]

print(f"\n=== 100% Score Problems ({len(perfect_df)} problems) ===")
if len(perfect_df) > 0:
    for _, r in perfect_df.iterrows():
        b_correct = "PASS" if r['Baseline_PerfectCorrect'] else f"FAIL ({r['Baseline_GAP_Count']} gaps)"
        e_correct = "PASS" if r['Enriched_PerfectCorrect'] else f"FAIL ({r['Enriched_GAP_Count']} gaps)"
        print(f"  Problem {r['ProblemID']}: Baseline={b_correct}, Enriched={e_correct}")

print(f"\n=== Failing Problems ({len(imperfect_df)} problems) ===")
print(f"Student's weak skills: {weak_skill_names}")

if len(imperfect_df) > 0:
    for _, r in imperfect_df.iterrows():
        b_tags = ", ".join(r['Baseline_KCTags']) if r['Baseline_KCTags'] else "(none)"
        e_tags = ", ".join(r['Enriched_KCTags']) if r['Enriched_KCTags'] else "(none)"
        match = "SAME" if r['Baseline_KCTags'] == r['Enriched_KCTags'] else "DIFF"
        print(f"  Problem {r['ProblemID']}: Baseline={b_tags} | Enriched={e_tags} [{match}]")

print(f"\n=== Aggregate Metrics (Failing Problems) ===")
if len(imperfect_df) > 0:
    b_f1 = imperfect_df['Baseline_WeakOverlap_F1'].mean()
    e_f1 = imperfect_df['Enriched_WeakOverlap_F1'].mean()
    print(f"  Avg WeakOverlap F1: Baseline={b_f1:.3f}, Enriched={e_f1:.3f}, Delta={e_f1-b_f1:+.3f}")

COMPARISON: Baseline vs Context-Enriched

=== 100% Score Problems (4 problems) ===
  Problem 1: Baseline=PASS, Enriched=PASS
  Problem 20: Baseline=PASS, Enriched=PASS
  Problem 232: Baseline=FAIL (2 gaps), Enriched=FAIL (2 gaps)
  Problem 233: Baseline=PASS, Enriched=PASS

=== Failing Problems (8 problems) ===
Student's weak skills: ['While', 'DefFunction', 'StringConcat', 'Math%', 'StringLen', 'StringIndex', 'StringEqual', 'ArrayIndex', 'StringFormat', 'For', 'NestedFor', 'LogicCompareNum', 'Math+-*/', 'If/Else']
  Problem 235: Baseline=If/Else, LogicAndNotOr, LogicBoolean, NestedIf | Enriched=If/Else, LogicAndNotOr, NestedIf [DIFF]
  Problem 236: Baseline=If/Else, LogicAndNotOr, NestedIf | Enriched=If/Else, LogicAndNotOr, NestedIf [SAME]
  Problem 106: Baseline=ArrayIndex, For, If/Else, LogicBoolean, NestedFor | Enriched=ArrayIndex, DefFunction, For, If/Else, LogicCompareNum, NestedFor [DIFF]
  Problem 46: Baseline=ArrayIndex, DefFunction, For, If/Else, LogicAndNotOr, NestedFor | En

In [ ]:
import json
from datetime import datetime
from pathlib import Path

# Ensure ROOT exists even when this cell is run standalone
ROOT = globals().get('ROOT', Path.cwd())
if not (ROOT / 'lib').exists() and (ROOT.parent / 'lib').exists():
    ROOT = ROOT.parent

# Resolve student id even if config cell was not run
if 'TARGET_STUDENT_ID' not in globals():
    TARGET_STUDENT_ID = int(comparison_df['SubjectID'].iloc[0]) if 'comparison_df' in globals() and len(comparison_df) > 0 else None

# --- Configuration ---
MODEL_VERSION = "v1_simple_average"  # CHANGE to "v2_difficulty_weighted" when rerunning
RESULTS_BASE = ROOT / 'results' / '05_mental_model_comparison'

# --- Create directory ---
results_dir = RESULTS_BASE / MODEL_VERSION
results_dir.mkdir(parents=True, exist_ok=True)

# --- Save results CSV ---
csv_path = results_dir / f'student_{TARGET_STUDENT_ID}.csv'
comparison_df.to_csv(csv_path, index=False)

# --- Save metadata ---
failing_df = comparison_df[comparison_df['IsPerfect'] == False]
perfect_df = comparison_df[comparison_df['IsPerfect'] == True]

metadata = {
    "experiment": "05_mental_model_comparison",
    "version": MODEL_VERSION,
    "run_timestamp": datetime.now().isoformat(),
    "model_id": MODEL_ID,
    "temperature": 0.3,
    "student_id": TARGET_STUDENT_ID,
    "num_problems": len(comparison_df),
    "problems_selected": comparison_df['ProblemID'].tolist(),
    "mental_model_type": MODEL_VERSION.replace("v1_", "").replace("v2_", ""),
    "weak_skill_threshold": 0.6,
    "weak_skills": [s[0] for s in weak_skills],
    "weak_skill_scores": {s[0]: round(s[1], 4) for s in weak_skills},
    "num_weak_skills": len(weak_skills),
    "score_distribution": {
        "perfect": int(len(perfect_df)),
        "partial": int(((comparison_df['ScorePct'] > 0) & (comparison_df['ScorePct'] < 99.9)).sum()),
        "zero": int((comparison_df['ScorePct'] <= 0).sum()),
    },
    "results_summary": {
        "baseline_avg_f1": round(float(failing_df['Baseline_WeakOverlap_F1'].mean()), 4) if len(failing_df) > 0 else 0.0,
        "enriched_avg_f1": round(float(failing_df['Enriched_WeakOverlap_F1'].mean()), 4) if len(failing_df) > 0 else 0.0,
        "delta_f1": round(float(failing_df['Enriched_WeakOverlap_F1'].mean() - failing_df['Baseline_WeakOverlap_F1'].mean()), 4) if len(failing_df) > 0 else 0.0,
        "perfect_baseline_correct": int(perfect_df['Baseline_PerfectCorrect'].sum()) if len(perfect_df) > 0 else 0,
        "perfect_enriched_correct": int(perfect_df['Enriched_PerfectCorrect'].sum()) if len(perfect_df) > 0 else 0,
        "perfect_total": int(len(perfect_df)),
    }
}

meta_path = results_dir / f'metadata_{TARGET_STUDENT_ID}.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"Results saved to: {results_dir}")
print(f"  CSV: {csv_path.name}")
print(f"  Metadata: {meta_path.name}")
print(f"\nQuick summary:")
print(f"  Model version: {MODEL_VERSION}")
print(f"  Student: {TARGET_STUDENT_ID}")
print(f"  Weak skills: {metadata['weak_skills']}")
print(f"  Baseline F1: {metadata['results_summary']['baseline_avg_f1']}")
print(f"  Enriched F1: {metadata['results_summary']['enriched_avg_f1']}")
print(f"  Delta F1: {metadata['results_summary']['delta_f1']}")